# Phase 2 — Netback Calculator

The aims doc's central message: **the highest quoted price is not the same as the highest economic value** once shipping, liquefaction, and other costs are priced in. This notebook computes netback for every origin/destination combination in the network (`src/network_config.py`) using `src/netback.py`, first on the latest real prices, then on simulated future scenarios from Phase 1's calibrated model.

    Netback = P_dest - C_shipping - C_liquefaction - C_variable - C_hedging

Cost assumptions (liquefaction, variable, hedging, FX, port/loading days) are illustrative placeholders -- see `CostAssumptions` in `src/netback.py` for the reasoning behind each. These are exactly the knobs a later dashboard would expose as adjustable sliders.

In [ ]:
import csv
import dataclasses
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, "../src")
from netback import CostAssumptions, compute_netback_all_destinations
from network_config import DESTINATIONS, ORIGINS
from simulator import HubParams, simulate_paths

## Load latest real prices and calibration

In [ ]:
with open("../data/processed/combined/monthly_aligned.csv") as f:
    rows = list(csv.DictReader(f))
latest = rows[-1]
print(f"Latest data point: {latest['date']} (TTF reconstructed: {latest['ttf_is_reconstructed']})")

current_prices = {
    "HENRY_HUB": float(latest["henry_hub"]),
    "TTF": float(latest["ttf"]),
    "JKM": float(latest["jkm"]),
}
current_freight = {
    "FREIGHT_ATLANTIC": float(latest["freight_atlantic"]),
    "FREIGHT_PACIFIC": float(latest["freight_pacific"]),
}
print("Prices:", current_prices)
print("Freight:", current_freight)

with open("../data/processed/calibration/hub_params.json") as f:
    calibration = json.load(f)

## Netback from every origin, on latest real prices

In [ ]:
assumptions = CostAssumptions()
rows_out = []
for origin in ORIGINS:
    for r in compute_netback_all_destinations(origin, current_prices, current_freight, assumptions):
        rows_out.append({
            "origin": origin,
            "destination": r.destination,
            "headline_price": round(r.p_dest_usd_per_mmbtu, 2),
            "shipping": round(r.c_shipping_usd_per_mmbtu, 2),
            "liquefaction": round(r.c_liquefaction_usd_per_mmbtu, 2),
            "variable": round(r.c_variable_usd_per_mmbtu, 2),
            "hedging": round(r.c_hedging_usd_per_mmbtu, 2),
            "netback": round(r.netback_usd_per_mmbtu, 2),
        })

df = pd.DataFrame(rows_out)
df

## The headline-price-vs-netback flip

For every origin, rank destinations by headline price alone vs. by netback -- do they agree?

In [ ]:
for origin in ORIGINS:
    sub = df[df["origin"] == origin]
    by_headline = sub.sort_values("headline_price", ascending=False)["destination"].tolist()
    by_netback = sub.sort_values("netback", ascending=False)["destination"].tolist()
    flip = "  <-- FLIPS" if by_headline != by_netback else ""
    print(f"{origin:<10} headline-best order: {by_headline}   netback-best order: {by_netback}{flip}")

Even where the ranking doesn't flip, the *margin* often does -- e.g. Australia's shipping cost to Europe (very long Cape voyage) is markedly higher than its shipping cost to Asia, so the gap between the two destinations narrows a lot after netback even without reordering. Whether headline price alone would have led to a different call depends entirely on the live TTF-JKM spread at decision time -- rerun the cell above with a different `current_prices` dict (or a simulated scenario below) to see the ranking flip when the spread narrows or reverses.

## Netback under a simulated future scenario

Draw one path from Phase 1's calibrated simulator and recompute netback at a future month -- this is exactly what Phase 3's optimization will consume: netback isn't a single number, it's a distribution over scenarios.

In [ ]:
hub_order = calibration["correlation"]["hub_order"]
corr_matrix = calibration["correlation"]["matrix"]
all_params = [HubParams(**calibration["primary"][name]) for name in hub_order]
start_prices = {
    "HENRY_HUB": current_prices["HENRY_HUB"],
    "TTF": current_prices["TTF"],
    "JKM": current_prices["JKM"],
    "FREIGHT_ATLANTIC": current_freight["FREIGHT_ATLANTIC"],
    "FREIGHT_PACIFIC": current_freight["FREIGHT_PACIFIC"],
}

N_PATHS = 2000
N_STEPS = 6  # 6 months ahead
paths = simulate_paths(all_params, corr_matrix, n_paths=N_PATHS, n_steps=N_STEPS, start_prices=start_prices, seed=123)

# For each of the N_PATHS simulated 6-month-ahead scenarios, compute netback
# for every origin/destination and track how often each destination wins.
win_counts = {origin: {d: 0 for d in ["EUROPE", "ASIA"] if d in [r.destination for r in compute_netback_all_destinations(origin, current_prices, current_freight)]} for origin in ORIGINS}

for i in range(N_PATHS):
    scenario_prices = {
        "HENRY_HUB": paths["HENRY_HUB"][i, -1],
        "TTF": paths["TTF"][i, -1],
        "JKM": paths["JKM"][i, -1],
    }
    scenario_freight = {
        "FREIGHT_ATLANTIC": paths["FREIGHT_ATLANTIC"][i, -1],
        "FREIGHT_PACIFIC": paths["FREIGHT_PACIFIC"][i, -1],
    }
    for origin in ORIGINS:
        results = [r for r in compute_netback_all_destinations(origin, scenario_prices, scenario_freight) if r.destination in ("EUROPE", "ASIA")]
        if not results:
            continue
        best = max(results, key=lambda r: r.netback_usd_per_mmbtu)
        win_counts[origin][best.destination] += 1

print(f"Across {N_PATHS} simulated 6-month-ahead scenarios, how often each destination wins on netback:\n")
for origin, counts in win_counts.items():
    total = sum(counts.values())
    parts = ", ".join(f"{d}: {c/total:.0%}" for d, c in counts.items())
    print(f"{origin:<10} {parts}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
origins_list = list(win_counts.keys())
europe_pct = [win_counts[o].get("EUROPE", 0) / sum(win_counts[o].values()) * 100 for o in origins_list]
asia_pct = [win_counts[o].get("ASIA", 0) / sum(win_counts[o].values()) * 100 for o in origins_list]
x = range(len(origins_list))
ax.bar(x, europe_pct, label="Europe wins", color="steelblue")
ax.bar(x, asia_pct, bottom=europe_pct, label="Asia wins", color="indianred")
ax.set_xticks(list(x))
ax.set_xticklabels(origins_list)
ax.set_ylabel("% of 2000 simulated 6-month scenarios")
ax.set_title("Which destination wins on netback -- by scenario, not by today's snapshot")
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

This is the real payoff of simulating scenarios instead of using a single point forecast: the "best" destination isn't fixed, it's a probability that shifts with the TTF-JKM spread -- exactly the kind of uncertainty a naive highest-price rule ignores, and exactly what Phase 3's risk-aware LP will need to weigh.